In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import IntSlider, FloatSlider, VBox, HBox, Layout, HTML
from IPython.display import display

plt.ioff()

# ==============================================================================
# USAGE
#
# This interactive notebook explores the behavior of a continuous-time
# low-pass Bessel-Thomson filter.
#
# Two parameters can be varied:
#
#       N     : filter order
#       τ0    : group delay at ω = 0
#
# The Bessel filter is constructed directly with scipy.signal.bessel using
#
#       norm='delay'
#
# so that the filter is normalized with respect to group delay rather than
# the -3 dB cutoff frequency.
#
# For the non-normalized case:
#
#       Wn = 1 / τ0
#
# and therefore
#
#       τ(0) = τ0.
#
# The notebook displays four responses:
#
# 1. Magnitude response |H(jω)|
#
#    The Bessel filter behaves as a low-pass filter with monotonic magnitude
#    response and a relatively wide transition region.
#
#    Unlike Butterworth, Chebyshev and elliptic filters, its principal design
#    objective is not sharp frequency selectivity.
#
# 2. Phase response ∠H(jω)
#
#    The red curve is the actual Bessel phase response.
#
#    The black dashed curve represents the ideal linear-phase relation
#
#       φideal(ω) = -ωτ0.
#
#    The vertical axis is scaled ONLY from the actual phase response.
#    Therefore, the ideal reference line does not compress the actual
#    Bessel phase curve.
#
# 3. Impulse response h(t)
#
#    The impulse response becomes smoother as the Bessel behavior approaches
#    the ideal constant-delay characteristic.
#
# 4. Step response
#
#    Bessel filters exhibit very small overshoot compared with filter
#    approximations optimized primarily for sharp frequency selectivity.
#
# Increasing N extends the frequency region over which the phase is nearly
# linear and the group delay is nearly constant.
#
# Increasing τ0 slows the system and stretches the impulse and step responses
# along the time axis.
#
# IMPORTANT CHARACTERISTIC
#
# Butterworth:
#       maximally flat magnitude response.
#
# Chebyshev I:
#       equiripple passband.
#
# Chebyshev II:
#       equiripple stopband.
#
# Elliptic / Cauer:
#       equiripple passband and stopband.
#
# Bessel-Thomson:
#       maximally flat group delay around ω = 0.
# ==============================================================================

# ==============================================================================
# JUPYTER DISPLAY SETTINGS
# ==============================================================================

display(HTML("""
<style>
.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.output,
.output_area,
.output_subarea,
.output_scroll {
    overflow: visible !important;
    max-height: none !important;
    height: auto !important;
}

.jupyter-widgets,
.widget-box,
.widget-html,
.widget-html-content {
    overflow: visible !important;
    max-height: none !important;
}

.jp-Cell-outputWrapper {
    overflow: visible !important;
}
</style>
"""))

# ==============================================================================
# DESCRIPTION
# ==============================================================================

description = HTML("""
<div style="
    border:1px solid #9ec9f5;
    border-radius:7px;
    padding:8px 10px;
    margin:0px 0px 7px 0px;
    font-size:13px;
    line-height:1.45;
    background-color:#f7fbff;
    width:810px;
    max-width:810px;
    box-sizing:border-box;
">
<b>Purpose:</b>
Explore the frequency-domain and time-domain behavior of a continuous-time low-pass Bessel-Thomson filter.
<br>
<b>Interpretation:</b>
The sliders control the filter order N and the zero-frequency group delay τ₀. Bessel filters are designed for maximally flat group delay near ω = 0 rather than for a sharp transition band. As N increases, the actual phase remains approximately linear over a wider frequency range, producing reduced waveform distortion and very small transient overshoot.
</div>
""", layout=Layout(width='820px', max_width='820px'))

# ==============================================================================
# CONTROLS
# ==============================================================================

slider_layout = Layout(width='250px')
style_opts = {'description_width':'75px'}

order_slider = IntSlider(min=1, max=10, step=1, value=4, description='Order N:', continuous_update=True, style=style_opts, layout=slider_layout)

tau_slider = FloatSlider(min=0.5, max=3.0, step=0.1, value=1.0, description='τ₀:', continuous_update=True, readout=True, readout_format='.1f', style=style_opts, layout=slider_layout)

parameter_title = HTML("""
<div style="
    font-size:14px;
    font-weight:bold;
    margin-top:3px;
    margin-bottom:5px;
">
Filter Parameters:
</div>
""")

info_html = HTML(layout=Layout(width='275px', max_width='275px'))

# ==============================================================================
# FIGURE 1: MAGNITUDE RESPONSE
# ==============================================================================

fig_mag, ax_mag = plt.subplots(figsize=(5.2, 3.15))

mag_line, = ax_mag.plot([], [], 'r-', linewidth=2.0, label='|H(jω)|')

dc_mag_line = ax_mag.axhline(1.0, color='gray', linestyle='--', linewidth=0.9, label='DC gain = 1')

ax_mag.set_xscale('log')

ax_mag.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_mag.set_ylabel('|H(jω)|', fontsize=10)

ax_mag.set_title('Bessel Magnitude Response', fontsize=12, fontweight='bold', pad=5)

ax_mag.tick_params(axis='both', labelsize=9)

ax_mag.grid(True, which='both', linestyle=':', alpha=0.5)

ax_mag.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

ax_mag.set_xlim(0.05, 50.0)
ax_mag.set_ylim(0.0, 1.08)

fig_mag.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_mag.canvas.header_visible = False
fig_mag.canvas.toolbar_visible = False
fig_mag.canvas.resizable = False

fig_mag.canvas.layout.width = '520px'
fig_mag.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 2: PHASE RESPONSE
# ==============================================================================

fig_phase, ax_phase = plt.subplots(figsize=(5.2, 3.15))

phase_line, = ax_phase.plot([], [], 'r-', linewidth=2.0, label='Actual phase')

ideal_phase_line, = ax_phase.plot([], [], 'k--', linewidth=1.2, label='Ideal −ωτ₀')

zero_phase_line = ax_phase.axhline(0.0, color='gray', linestyle=':', linewidth=0.8)

ax_phase.set_xscale('log')

ax_phase.set_xlabel('Angular Frequency ω (rad/s)', fontsize=10)
ax_phase.set_ylabel('Phase (degrees)', fontsize=10)

ax_phase.set_title('Bessel Phase Response', fontsize=12, fontweight='bold', pad=5)

ax_phase.tick_params(axis='both', labelsize=9)

ax_phase.grid(True, which='both', linestyle=':', alpha=0.5)

ax_phase.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

ax_phase.set_xlim(0.05, 50.0)

fig_phase.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_phase.canvas.header_visible = False
fig_phase.canvas.toolbar_visible = False
fig_phase.canvas.resizable = False

fig_phase.canvas.layout.width = '520px'
fig_phase.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 3: IMPULSE RESPONSE
# ==============================================================================

fig_impulse, ax_impulse = plt.subplots(figsize=(5.2, 3.15))

impulse_line, = ax_impulse.plot([], [], 'r-', linewidth=2.0, label='h(t)')

zero_impulse_line = ax_impulse.axhline(0.0, color='gray', linestyle='--', linewidth=0.8)

ax_impulse.set_xlabel('Time t (s)', fontsize=10)
ax_impulse.set_ylabel('h(t)', fontsize=10)

ax_impulse.set_title('Bessel Impulse Response', fontsize=12, fontweight='bold', pad=5)

ax_impulse.tick_params(axis='both', labelsize=9)

ax_impulse.grid(True, linestyle=':', alpha=0.5)

ax_impulse.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=1, fontsize=8)

ax_impulse.set_xlim(0.0, 24.0)
ax_impulse.set_xticks([0, 4, 8, 12, 16, 20, 24])

ax_impulse.set_ylim(-0.5, 2.5)
ax_impulse.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0, 2.5])

fig_impulse.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_impulse.canvas.header_visible = False
fig_impulse.canvas.toolbar_visible = False
fig_impulse.canvas.resizable = False

fig_impulse.canvas.layout.width = '520px'
fig_impulse.canvas.layout.height = '320px'

# ==============================================================================
# FIGURE 4: STEP RESPONSE
# ==============================================================================

fig_step, ax_step = plt.subplots(figsize=(5.2, 3.15))

step_line, = ax_step.plot([], [], 'r-', linewidth=2.0, label='Step response')

final_value_line = ax_step.axhline(1.0, color='gray', linestyle='--', linewidth=1.0, label='Final value = 1')

ax_step.set_xlabel('Time t (s)', fontsize=10)
ax_step.set_ylabel('Amplitude', fontsize=10)

ax_step.set_title('Bessel Step Response', fontsize=12, fontweight='bold', pad=5)

ax_step.tick_params(axis='both', labelsize=9)

ax_step.grid(True, linestyle=':', alpha=0.5)

ax_step.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8)

ax_step.set_xlim(0.0, 24.0)
ax_step.set_xticks([0, 4, 8, 12, 16, 20, 24])

ax_step.set_ylim(-0.1, 1.5)
ax_step.set_yticks([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])

fig_step.subplots_adjust(left=0.13, right=0.97, bottom=0.31, top=0.86)

fig_step.canvas.header_visible = False
fig_step.canvas.toolbar_visible = False
fig_step.canvas.resizable = False

fig_step.canvas.layout.width = '520px'
fig_step.canvas.layout.height = '320px'

# ==============================================================================
# FIXED AXES
# ==============================================================================

omega = np.logspace(np.log10(0.05), np.log10(50.0), 3000)

t_max = 24.0

t = np.linspace(0.0, t_max, 3000)

# ==============================================================================
# UPDATE FUNCTION
# ==============================================================================

def update_bessel(change=None):

    N = order_slider.value
    tau0 = tau_slider.value

    # --------------------------------------------------------------------------
    # Frequency scaling
    # --------------------------------------------------------------------------

    Wn = 1.0 / tau0

    # --------------------------------------------------------------------------
    # Bessel-Thomson analog transfer function
    # --------------------------------------------------------------------------

    b, a = signal.bessel(N, Wn, btype='low', analog=True, output='ba', norm='delay')

    # --------------------------------------------------------------------------
    # Poles and zeros
    # --------------------------------------------------------------------------

    zeros, poles, gain = signal.bessel(N, Wn, btype='low', analog=True, output='zpk', norm='delay')

    # --------------------------------------------------------------------------
    # Frequency response
    # --------------------------------------------------------------------------

    _, H = signal.freqs(b, a, worN=omega)

    magnitude = np.abs(H)

    phase = np.unwrap(np.angle(H))

    phase_deg = np.rad2deg(phase)

    # --------------------------------------------------------------------------
    # Ideal linear phase corresponding to constant group delay τ0
    # --------------------------------------------------------------------------

    ideal_phase = -omega * tau0

    ideal_phase_deg = np.rad2deg(ideal_phase)

    # --------------------------------------------------------------------------
    # Numerical group delay
    # --------------------------------------------------------------------------

    group_delay = -np.gradient(phase, omega)

    tau_zero_numeric = group_delay[0]

    # --------------------------------------------------------------------------
    # Impulse and step responses
    # --------------------------------------------------------------------------

    system = signal.TransferFunction(b, a)

    t_impulse, h = signal.impulse(system, T=t)

    t_step, y_step = signal.step(system, T=t)

    # --------------------------------------------------------------------------
    # DC gain
    # --------------------------------------------------------------------------

    dc_gain = np.abs(b[-1] / a[-1])

    # --------------------------------------------------------------------------
    # Approximate -3 dB frequency
    # --------------------------------------------------------------------------

    target = 1.0 / np.sqrt(2.0)

    index_3db = np.argmin(np.abs(magnitude - target))

    omega_3db = omega[index_3db]

    # --------------------------------------------------------------------------
    # Magnitude response update
    # --------------------------------------------------------------------------

    mag_line.set_data(omega, magnitude)

    ax_mag.set_xlim(0.05, 50.0)

    ax_mag.set_ylim(0.0, 1.08)

    # --------------------------------------------------------------------------
    # Phase response update
    # --------------------------------------------------------------------------

    phase_line.set_data(omega, phase_deg)

    ideal_phase_line.set_data(omega, ideal_phase_deg)

    ax_phase.set_xlim(0.05, 50.0)

    # IMPORTANT:
    # The vertical scale is determined ONLY by the actual Bessel phase.
    # The ideal dashed line does not affect the y-axis range.

    phase_min = np.min(phase_deg)

    ax_phase.set_ylim(1.05 * phase_min, 5.0)

    # --------------------------------------------------------------------------
    # Impulse response update
    # --------------------------------------------------------------------------

    impulse_line.set_data(t_impulse, h)

    ax_impulse.set_xlim(0.0, 24.0)

    ax_impulse.set_xticks([0, 4, 8, 12, 16, 20, 24])

    ax_impulse.set_ylim(-0.5, 2.5)

    ax_impulse.set_yticks([-0.5, 0, 0.5, 1.0, 1.5, 2.0, 2.5])

    # --------------------------------------------------------------------------
    # Step response update
    # --------------------------------------------------------------------------

    step_line.set_data(t_step, y_step)

    final_value_line.set_ydata([dc_gain, dc_gain])

    ax_step.set_xlim(0.0, 24.0)

    ax_step.set_xticks([0, 4, 8, 12, 16, 20, 24])

    ax_step.set_ylim(-0.1, 1.5)

    ax_step.set_yticks([0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])

    # --------------------------------------------------------------------------
    # Poles
    # --------------------------------------------------------------------------

    pole_text = '<br>'.join([f'p{k + 1} = {p.real:+.4f} {p.imag:+.4f}j' for k, p in enumerate(poles)])

    # --------------------------------------------------------------------------
    # Information panel
    # --------------------------------------------------------------------------

    info_html.value = f"""
    <div style="
        border:1px solid #cccccc;
        border-radius:7px;
        padding:8px 9px;
        margin-top:9px;
        font-size:12px;
        line-height:1.65;
        background:white;
        width:270px;
        box-sizing:border-box;
    ">

    <div>
        <b>Filter:</b>
        <span style="color:#0066cc;">Bessel-Thomson low-pass</span>
    </div>

    <div>
        <b>Order N:</b>
        <span style="color:#0066cc;">{N}</span>
    </div>

    <div>
        <b>Target τ(0):</b>
        <span style="color:#0066cc;">{tau0:.2f} s</span>
    </div>

    <div>
        <b>Computed τ(0):</b>
        <span style="color:#0066cc;">{tau_zero_numeric:.4f} s</span>
    </div>

    <div>
        <b>Frequency scale Wn:</b>
        <span style="color:#0066cc;">{Wn:.4f} rad/s</span>
    </div>

    <div>
        <b>Approx. −3 dB frequency:</b>
        <span style="color:#0066cc;">{omega_3db:.4f} rad/s</span>
    </div>

    <div>
        <b>DC gain:</b>
        <span style="color:#0066cc;">{dc_gain:.4f}</span>
    </div>

    <div>
        <b>Finite zeros:</b>
        <span style="color:#0066cc;">None</span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Poles:</b><br>
        <span style="color:#0066cc;">
        {pole_text}
        </span>
    </div>

    <div style="
        margin-top:6px;
        padding-top:6px;
        border-top:1px solid #eeeeee;
    ">
        <b>Observation:</b><br>
        Increasing N extends the approximately linear-phase region and the
        approximately constant-group-delay region. Increasing τ₀ stretches
        the time-domain responses and shifts the frequency response toward
        lower frequencies.
    </div>

    </div>
    """

    # --------------------------------------------------------------------------
    # REDRAW EXISTING FIGURES ONLY
    # --------------------------------------------------------------------------

    fig_mag.canvas.draw_idle()

    fig_phase.canvas.draw_idle()

    fig_impulse.canvas.draw_idle()

    fig_step.canvas.draw_idle()

# ==============================================================================
# CALLBACKS
# ==============================================================================

order_slider.observe(update_bessel, names='value')

tau_slider.observe(update_bessel, names='value')

# ==============================================================================
# LAYOUT: 2 x 2 FIGURE GRID
# ==============================================================================

controls = VBox([parameter_title, order_slider, tau_slider, info_html], layout=Layout(width='285px', min_width='285px', max_width='285px', flex='0 0 285px', align_items='flex-start'))

top_row = HBox([fig_mag.canvas, fig_phase.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

bottom_row = HBox([fig_impulse.canvas, fig_step.canvas], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

plot_grid = VBox([top_row, bottom_row], layout=Layout(width='1050px', align_items='flex-start', justify_content='flex-start'))

main_layout = HBox([controls, plot_grid], layout=Layout(width='1340px', align_items='flex-start', justify_content='flex-start'))

# ==============================================================================
# INITIALIZE DATA
# ==============================================================================

update_bessel()

# ==============================================================================
# DISPLAY
# ==============================================================================

display(description)

display(main_layout)